<a href="https://colab.research.google.com/github/GermanVelasco19/Premier-League-Project/blob/main/PremierLeaguePredict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##IMPORTAR LAS LIBRERIAS A USAR

In [ ]:
!pip install statsbombpy
!apt-get update -q
!apt-get install -y chromium-browser chromium-chromedriver -q
!pip install soccerdata -q
!pip install pandas requests -q

In [4]:
#CELDA PARA IMPORTAR LIBRERIAS
import pandas as pd
import numpy as np
import gdown
import soccerdata as sd
from statsbombpy import sb
import sys
import os
import requests
import time

sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

##DESCARGAR LOS ARCHIVOS DE LAS CARPETA CON LA INFORMACIÓN DE LOS PARTIDOS DESDE LA TEMPORADA 2000-2001

In [5]:
# URLs directas por temporada - Premier League
temporadas = {
    "2018-2019": "https://www.football-data.co.uk/mmz4281/1819/E0.csv",
    "2019-2020": "https://www.football-data.co.uk/mmz4281/1920/E0.csv",
    "2020-2021": "https://www.football-data.co.uk/mmz4281/2021/E0.csv",
    "2021-2022": "https://www.football-data.co.uk/mmz4281/2122/E0.csv",
    "2022-2023": "https://www.football-data.co.uk/mmz4281/2223/E0.csv",
    "2023-2024": "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "2024-2025": "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
    "2025-2026": "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
}

dfs = []

for temporada, url in temporadas.items():
    try:
        df = pd.read_csv(url)
        df["season"] = temporada
        dfs.append(df)
        print(f"✓ {temporada}: {len(df)} partidos")
        time.sleep(1)
    except Exception as e:
        print(f"✗ {temporada}: {e}")

# Unir todas las temporadas
df_raw = pd.concat(dfs, ignore_index=True)

print(f"\nTotal filas: {df_raw.shape[0]}")
print(f"Columnas disponibles: {df_raw.columns.tolist()}")
df_raw.head()

✓ 2018-2019: 380 partidos
✓ 2019-2020: 380 partidos
✓ 2020-2021: 380 partidos
✓ 2021-2022: 380 partidos
✓ 2022-2023: 380 partidos
✓ 2023-2024: 380 partidos
✓ 2024-2025: 380 partidos
✓ 2025-2026: 359 partidos

Total filas: 3019
Columnas disponibles: ['Div', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5', 'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA', 'PSCH', 'PSCD', 'PSCA', 'season', 'Time', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'B365CH

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,E0,10/08/2018,Man United,Leicester,2,1,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,11/08/2018,Bournemouth,Cardiff,2,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,11/08/2018,Fulham,Crystal Palace,0,2,A,0,1,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,11/08/2018,Huddersfield,Chelsea,0,3,A,0,2,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,11/08/2018,Newcastle,Tottenham,1,2,A,1,2,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##PREPARACIÓN DE LOS DATOS

In [6]:
print(df_raw.columns.tolist())
print(df_raw.shape)
df_raw.head(3)

['Div', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5', 'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA', 'PSCH', 'PSCD', 'PSCA', 'season', 'Time', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'IWCH', 'IWCD', 'IWCA', 'WHCH', 'WHCD', 'WHCA', 'VCCH', 'VCCD', 'VCCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,E0,10/08/2018,Man United,Leicester,2,1,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,11/08/2018,Bournemouth,Cardiff,2,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,11/08/2018,Fulham,Crystal Palace,0,2,A,0,1,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
columnas = ["season", "Date", "HomeTeam", "AwayTeam",
            "FTHG", "FTAG",        # Real Goals → Target variable
            "HTHG", "HTAG",        # Mid time Goals
            "HS", "AS",            # Total shots
            "HST", "AST",          # Shots on goal
            "HF", "AF",            # Fauls
            "HC", "AC"]
df = df_raw[columnas].copy()

In [9]:
df = df.dropna(subset=["FTHG", "FTAG", "HomeTeam", "AwayTeam"])
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.sort_values("Date").reset_index(drop=True)

In [10]:
df["target_home_goals"] = df["FTHG"].astype(int)
df["target_away_goals"] = df["FTAG"].astype(int)